## 1️⃣ Initial Setup and Data Download

**Task**: Download the BBC News summarization dataset and explore its structure.

**Requirements**:
- Download dataset using KaggleHub
- Explore directory structure
- Understand the organization of articles and summaries

In [1]:
import kagglehub
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import re

# TODO: Download the BBC News Summary dataset
path = kagglehub.dataset_download("pariza/bbc-news-summary")

# TODO: Create function to explore dataset structure
def explore_dataset_structure(base_path):
    base = Path(base_path)

    # locate dataset root
    dataset_root = base / "BBC News Summary"
    articles_root = dataset_root / "News Articles"
    summaries_root = dataset_root / "Summaries"

    # get all article categories
    categories = [d.name for d in articles_root.iterdir() if d.is_dir()]

    # count total articles across all categories
    total = sum(len(list((articles_root / cat).glob("*.txt"))) for cat in categories)

    return articles_root, summaries_root, categories, total

# TODO: Explore the dataset structure
articles_root, summaries_root, categories, total_articles = explore_dataset_structure(path)

# TODO: Load and display sample article and summary
sample_cat = categories[0]
sample_article_path = list((articles_root / sample_cat).glob("*.txt"))[0]
sample_summary_path = summaries_root / sample_cat / sample_article_path.name

# TODO: Read sample article
article_text = sample_article_path.read_text(encoding="latin-1")

# TODO: Read corresponding summary
summary_text = sample_summary_path.read_text(encoding="latin-1")

# TODO: Display basic statistics
article_len = len(article_text.split())
summary_len = len(summary_text.split())

print("Total articles loaded:", total_articles)
print("Sample article length:", article_len, "words")
print("Sample summary length:", summary_len, "words")


Total articles loaded: 2225
Sample article length: 538 words
Sample summary length: 192 words


## 2️⃣ Data Loading and Preprocessing

**Task**: Create a comprehensive dataset class to load and clean all articles and summaries.

**Requirements**:
- Implement text cleaning function
- Load all articles and summaries from all categories
- Create paired article-summary dataset
- Handle encoding issues and missing files
- Display dataset statistics and category distribution

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm
import numpy as np

In [3]:
# TODO: Create comprehensive dataset class for BBC News
class BBCNewsDataset:
    def __init__(self, base_path):
        # TODO: Initialize paths and load data
        from pathlib import Path
        self.base = Path(base_path)
        root = self.base / "BBC News Summary"
        if not root.exists():
            # fallback: search recursively if needed
            for c in self.base.rglob("BBC News Summary"):
                if (c / "News Articles").exists() and (c / "Summaries").exists():
                    root = c
                    break
        self.articles_root = root / "News Articles"
        self.summaries_root = root / "Summaries"
        self.data = self.load_all_data()

    def clean_text(self, text):
        # TODO: Implement basic text cleaning
        text = text.strip()
        # TODO: Remove extra whitespace
        import re
        text = re.sub(r"\s+", " ", text)
        return text

    def load_all_data(self):
        # TODO: Load all articles and summaries from all categories
        from pathlib import Path
        data = []
        # iterate categories
        for cat_dir in sorted(p for p in self.articles_root.iterdir() if p.is_dir()):
            cat = cat_dir.name
            # TODO: Get all article files in category
            for art_fp in sorted(cat_dir.glob("*.txt")):
                sum_fp = self.summaries_root / cat / art_fp.name
                if not sum_fp.exists():
                    continue
                # TODO: Read article text
                try:
                    article = art_fp.read_text(encoding="utf-8")
                except UnicodeDecodeError:
                    article = art_fp.read_text(encoding="latin-1", errors="replace")
                # TODO: Read corresponding summary
                try:
                    summary = sum_fp.read_text(encoding="utf-8")
                except UnicodeDecodeError:
                    summary = sum_fp.read_text(encoding="latin-1", errors="replace")
                # TODO: Clean both texts
                article = self.clean_text(article)
                summary = self.clean_text(summary)
                # TODO: Only include valid pairs
                if article and summary:
                    data.append({
                        "category": cat,
                        "filename": art_fp.name,
                        "article": article,
                        "summary": summary
                    })
        return data

# TODO: Load the dataset
ds = BBCNewsDataset(path)

# TODO: Convert to pandas DataFrame for easier handling
df = pd.DataFrame(ds.data, columns=["category", "filename", "article", "summary"])

# TODO: Display dataset statistics
print(f"Loaded {len(df)} article-summary pairs")
print(f"Dataset shape: {df.shape}")
print("Categories:", df["category"].name)
print(df["category"].value_counts().to_string())
print(df["category"].value_counts().dtype)


Loaded 2225 article-summary pairs
Dataset shape: (2225, 4)
Categories: category
category
sport            511
business         510
politics         417
tech             401
entertainment    386
int64


## 3️⃣ Model Initialization and Setup

**Task**: Initialize the BART model and tokenizer for sequence-to-sequence learning.

**Requirements**:
- Load pre-trained BART-base model and tokenizer
- Set up device configuration (GPU/CPU)
- Display model information and parameter count
- Prepare model for fine-tuning

In [4]:
# TODO: Initialize BART model and tokenizer
import torch
from transformers import BartForConditionalGeneration, BartTokenizerFast

model_name = "facebook/bart-base"
tokenizer = BartTokenizerFast.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# TODO: Check if GPU is available and move model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# TODO: Display model information
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Device: {device.type}")
print(f"Model parameters: {total_params:,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Model: facebook/bart-base
Device: cuda
Model parameters: 139,420,416


## 4️⃣ Data Preprocessing and Tokenization

**Task**: Prepare the dataset for BART training by tokenizing inputs and targets.

**Requirements**:
- Implement preprocessing function for sequence-to-sequence data
- Tokenize articles (inputs) and summaries (targets)
- Set appropriate maximum lengths for input and target sequences
- Create train-validation split
- Apply preprocessing to both splits

In [5]:
# TODO: Define preprocessing function for sequence-to-sequence data
def preprocess_function(examples, max_input_length=1024, max_target_length=128):
    # TODO: Tokenize inputs (articles)
    model_inputs = tokenizer(
        examples["article"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )
    # TODO: Tokenize targets (summaries)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["summary"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )
    # TODO: Add labels to model inputs
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# TODO: Convert DataFrame to Hugging Face Dataset
from datasets import Dataset
raw_ds = Dataset.from_pandas(df[["article", "summary"]])

# TODO: Split dataset into train and validation
split_ds = raw_ds.train_test_split(test_size=0.1, seed=42, shuffle=True)

# TODO: Apply preprocessing to both datasets
train_dataset = split_ds["train"].map(
    preprocess_function, batched=True, remove_columns=split_ds["train"].column_names
)
eval_dataset = split_ds["test"].map(
    preprocess_function, batched=True, remove_columns=split_ds["test"].column_names
)

# TODO: Display dataset sizes
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")


Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Train dataset size: 2002
Eval dataset size: 223


## 5️⃣ Text Generation Function

**Task**: Implement a function to generate summaries using the BART model.

**Requirements**:
- Create generation function with proper parameters
- Handle tokenization and device placement
- Implement beam search for better quality
- Add length penalties and early stopping
- Return decoded summary text

In [6]:
def generate_summary(text, max_length=128, min_length=30):
    # TODO: Tokenize input text
    inputs = tokenizer(
        [text],
        max_length=1024,
        truncation=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # TODO: Generate summary with beam search
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            num_beams=4,
            max_length=max_length,
            min_length=min_length,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    # TODO: Decode summary to text
    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )


## 6️⃣ Pre-training Evaluation

**Task**: Test the pre-trained BART model before fine-tuning to establish baseline performance.

**Requirements**:
- Generate summaries for sample articles using pre-trained model
- Compare generated summaries with ground truth
- Display examples from different categories
- Analyze baseline quality and identify areas for improvement

In [7]:
# TODO: Test on a few examples from the dataset before training
# TODO: Generate summaries for sample articles
# TODO: Compare with ground truth summaries
# TODO: Display results for analysis

def _pick_samples(df, k=3):
    # prefer different categories if possible, deterministic order
    samples = []
    for cat in sorted(df["category"].unique()):
        row = df[df["category"] == cat].iloc[0]
        samples.append(row)
        if len(samples) == k:
            break
    if len(samples) < k:
        # fill remaining deterministically
        extra = df.iloc[: (k - len(samples))].to_dict("records")
        samples.extend(extra)
    return samples[:k]

def _truncate(text, max_chars=1200):
    return (text[:max_chars].rstrip() + "...") if len(text) > max_chars else text

print("\nTesting on sample articles:")
print("=" * 80)
print()

for i, row in enumerate(_pick_samples(df, k=3), start=1):
    article = row["article"]
    gold = row["summary"]
    gen = generate_summary(article, max_length=128, min_length=30)

    print(f"Sample {i} - Category: {row['category']}")
    print(f"Article: {_truncate(article)}")
    print(f"Ground Truth: {gold}")
    print(f"Generated: {gen}")
    print("-" * 80)
    print()



Testing on sample articles:

Sample 1 - Category: business
Article: Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier. The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL. Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service

## 7️⃣ Training Configuration

**Task**: Set up training arguments and data collator for BART fine-tuning.

**Requirements**:
- Configure training arguments with appropriate hyperparameters
- Set up data collator for sequence-to-sequence training
- Configure learning rate, batch size, and number of epochs
- Enable mixed precision training if GPU available
- Set up logging and evaluation strategies

In [9]:
# TODO: Create data collator for sequence-to-sequence training
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
)

# TODO: Configure training arguments
# TODO: Set up learning rate, batch size, epochs
# TODO: Configure mixed precision, logging, evaluation
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetune",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.03,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to=["none"],  # disable external loggers like wandb
)

## 8️⃣ Evaluation Metrics Setup

**Task**: Configure ROUGE metrics for evaluating summarization quality.

**Requirements**:
- Load ROUGE evaluation metric
- Implement compute_metrics function for training
- Handle prediction decoding and label processing
- Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores
- Format results as percentages

In [19]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b215d962b7d4ac576c1c8db6c7c5edc22495f6629dd10a86d2231b2086f87e82
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [17]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [22]:
import evaluate

# TODO: Load ROUGE metric for summarization evaluation
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    # TODO: Decode predictions to text
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )

    # TODO: Replace -100 in the labels (can't decode padding tokens)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )

    # TODO: Compute ROUGE scores
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    result = rouge.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )

    # TODO: Convert to percentages
    return {
        "rouge1": round(result["rouge1"] * 100, 2),
        "rouge2": round(result["rouge2"] * 100, 2),
        "rougeL": round(result["rougeL"] * 100, 2),
    }


## 9️⃣ Trainer Initialization and Baseline Evaluation

**Task**: Initialize the Seq2SeqTrainer and evaluate pre-trained model performance.

**Requirements**:
- Create Seq2SeqTrainer with all components
- Evaluate baseline performance before fine-tuning
- Display baseline ROUGE scores
- Prepare for training process

In [23]:
# TODO: Evaluate the pretrained model first to get baseline scores
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

metrics = trainer.evaluate(max_length=128, num_beams=4)

# TODO: Display baseline ROUGE scores
print("Baseline ROUGE scores (before fine-tuning):")
print(f"eval_rouge1: {metrics.get('eval_rouge1'):.4f}")
print(f"eval_rouge2: {metrics.get('eval_rouge2'):.4f}")
print(f"eval_rougeL: {metrics.get('eval_rougeL'):.4f}")
print(f"eval_rougeLsum: {metrics.get('eval_rougeLsum', metrics.get('eval_rougeL')):.4f}")


/tmp/ipython-input-1273188305.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Baseline ROUGE scores (before fine-tuning):
eval_rouge1: 54.6900
eval_rouge2: 37.7900
eval_rougeL: 36.6300
eval_rougeLsum: 36.6300


## 🔟 Model Training

**Task**: Fine-tune the BART model on the BBC News dataset.

**Requirements**:
- Start the training process using the configured trainer
- Monitor training progress and loss
- Track evaluation metrics during training
- Save model checkpoints automatically
- Handle potential memory issues with appropriate batch sizes

In [24]:
# TODO: Start training the model
train_result = trainer.train()
print(train_result)


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3917: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=378, training_loss=0.4102103420035549, metrics={'train_runtime': 318.2637, 'train_samples_per_second': 18.871, 'train_steps_per_second': 1.188, 'total_flos': 3662077168189440.0, 'train_loss': 0.4102103420035549, 'epoch': 3.0})


## 1️⃣1️⃣ Model Saving and Loading

**Task**: Save the fine-tuned model and prepare for evaluation.

**Requirements**:
- Save the final trained model and tokenizer
- Load the fine-tuned model for evaluation
- Ensure model is moved to correct device
- Prepare for post-training evaluation

In [25]:
# TODO: Save the final model and tokenizer
save_dir = "./bart-bbc-final"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Load the fine-tuned model for evaluation
from transformers import BartForConditionalGeneration, BartTokenizerFast

finetuned_tokenizer = BartTokenizerFast.from_pretrained(save_dir)
finetuned_model = BartForConditionalGeneration.from_pretrained(save_dir)

# Ensure model is moved to correct device
finetuned_model.to(device)
finetuned_model.eval()


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [26]:
# TODO: Load the trained model for evaluation
from transformers import BartForConditionalGeneration, BartTokenizerFast

load_dir = "./bart-bbc-final"
tokenizer = BartTokenizerFast.from_pretrained(load_dir)
model = BartForConditionalGeneration.from_pretrained(load_dir)
model.to(device)
model.eval()


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

## 1️⃣2️⃣ Post-Training Evaluation

**Task**: Evaluate the fine-tuned model and compare with baseline performance.

**Requirements**:
- Generate summaries using the fine-tuned model
- Compare with ground truth summaries
- Show examples from different categories
- Test on the same samples used for baseline evaluation

In [27]:
# TODO: Test on the same examples after training
# TODO: Generate summaries using fine-tuned model
# TODO: Compare with ground truth and baseline
# TODO: Display improvement analysis

def _pick_same_samples(df, k=3):
    # reuse deterministic selection: first 3 rows (same split/seed -> same rows as baseline snippet)
    return [df.iloc[i] for i in range(min(k, len(df)))]

def _truncate(text, max_chars=300):
    return (text[:max_chars].rstrip() + " ...") if len(text) > max_chars else text

print("\nTesting on sample articles - After Training:")
print("=" * 80)
print()

for i, row in enumerate(_pick_same_samples(df, k=3), start=1):
    article = row["article"]
    gold = row["summary"]
    gen = generate_summary(article, max_length=128, min_length=30)

    print(f"Sample {i} - Category: {row['category']}")
    print(f"Article: {_truncate(article)}")
    print(f"Ground Truth: {gold}")
    print(f"Generated: {gen}")
    print("-" * 80)
    print()



Testing on sample articles - After Training:

Sample 1 - Category: business
Article: Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier. The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher ...
Ground Truth: TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn.For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.For 2005, TimeWarner

## 1️⃣3️⃣ Custom Article Testing

**Task**: Test the fine-tuned model on custom news articles to verify generalization.

**Requirements**:
- Create custom news article for testing
- Generate summary using fine-tuned model
- Analyze quality of generated summary
- Verify model's ability to generalize to new content
- Compare summary characteristics with training data

In [28]:
# TODO: Test with custom article
# TODO: Create custom news article for testing
custom_article = (
    "The government announced new policies to tackle climate change today.\n"
    "The prime minister said that renewable energy investments will increase by 50% next year.\n"
    "Solar and wind power projects will receive additional funding.\n"
    "Environmental groups welcomed the announcement but said more action is needed.\n"
    "The new policies include tax incentives for electric vehicle purchases and stricter emissions standards for industries.\n"
    "Coal power plants will be phased out over the next decade, while nuclear energy capacity will be expanded.\n"
    "Critics argue that the timeline is too slow to meet international climate commitments."
)

# TODO: Generate summary using fine-tuned model
generated = generate_summary(custom_article, max_length=128, min_length=30)

# TODO: Analyze quality and generalization
print("Custom Article:")
print(custom_article)
print("\nGenerated Summary:")
print(generated)


Custom Article:
The government announced new policies to tackle climate change today.
The prime minister said that renewable energy investments will increase by 50% next year.
Solar and wind power projects will receive additional funding.
Environmental groups welcomed the announcement but said more action is needed.
The new policies include tax incentives for electric vehicle purchases and stricter emissions standards for industries.
Coal power plants will be phased out over the next decade, while nuclear energy capacity will be expanded.
Critics argue that the timeline is too slow to meet international climate commitments.

Generated Summary:
The government announced new policies to tackle climate change today.The prime minister said that renewable energy investments will increase by 50% next year.The new policies include tax incentives for electric vehicle purchases and stricter emissions standards for industries.


## 📊 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (40%)**
   - Proper dataset loading and preprocessing
   - Correct BART model initialization and configuration
   - Working tokenization and data preparation pipeline
   - Successful model training and fine-tuning

2. **Model Performance (30%)**
   - Improvement in ROUGE scores after fine-tuning
   - Quality of generated summaries
   - Proper convergence during training
   - Meaningful comparison between pre-trained and fine-tuned models

3. **Code Quality and Analysis (30%)**
   - Clean, readable code with proper structure
   - Comprehensive evaluation and comparison
   - Good use of Hugging Face Transformers library
   - Proper handling of sequence-to-sequence training